In [1]:
#----------------------------------------------------------------------------------------------
#                                       Import Statements
#----------------------------------------------------------------------------------------------
import os 
import time 
from pinecone import Pinecone, ServerlessSpec
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

#----------------------------------------------------------------------------------------------
#                                       Init Statements
#----------------------------------------------------------------------------------------------
# Load environment variables from .env
load_dotenv()
# Fetch database URL
DATABASE_URL = os.getenv("DATABASE_URL")
api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)
spec = ServerlessSpec(
    cloud = "aws" , region = "us-east-1"
)
index_name = "shop-product-sample"

In [2]:
# Connect to pinecone 
myindex = pc.Index(index_name)

In [3]:
time.sleep(5)

In [4]:
embed_model = OpenAIEmbeddings(
    api_key=os.getenv("OPEN_AI_API"),
    model="text-embedding-3-small"
)

In [7]:
from langchain_pinecone import PineconeVectorStore
vectorsotre = PineconeVectorStore(
    index = myindex ,
    embedding=embed_model,
    text_key='Description'
)

In [8]:
query = "What is the price of SuperStar bold 2 product?"

vectorsotre.similarity_search(query , k=1)

[Document(id='1042', metadata={'Gender': 'Men', 'Price': '11999', 'PrimaryColor': 'Black', 'ProductBrand': 'Adidas', 'ProductName': 'Superstar Bold 2'}, page_content='Classic basketball shoes with a modern twist and bold design')]

In [29]:
from langchain_openai import ChatOpenAI
client = ChatOpenAI(api_key=os.getenv("OPEN_AI_API") , model="gpt-4o-mini")

chat_history = []

system_message = (
    "Act as an shop assistant and provide relevant response as per the user query"
)

In [30]:
def get_openai_resp(system_message , chat_history , prompt):
    # Append to the chathistory 
    chat_history.append(f"User : {prompt}")
    # Combine system message in chat history
    fullprompt = f"{system_message}\n\n" + "\n\n".join(chat_history)
    # Generate response 
    response = client.invoke(fullprompt)
    chat_history.append(f"Assistant : {response.content}")
    return response.content

In [31]:
def get_relevant_chunk(query , vectorstore):
    results = vectorstore.similarity_search(query , k =1 )
    if results:
        metadata = results[0].metadata
        context = (
            f"Product name : {metadata.get('ProductName' , 'not_available')}\n",
            f"Product brand : {metadata.get('ProductBrand' , 'not_available')}\n",
            f"Product price : {metadata.get('Price' , 'not_available')}\n",
            f"Product colour : {metadata.get('PrimaryColor' , 'not_available')}\n",
            f"Description : {results[0].page_content}\n"
        )
        return context 
    return "No relevant search" 

In [32]:
def make_prompt_chatbot(query,context):
    return f"Query : {query}\n\nContext : {context}\n\nAnswer:"

In [ ]:
def main():
    query = "What is the price of SuperStar bold 2 product? Just price no more text"
    relevant_text = get_relevant_chunk(query , vectorsotre)
    prompt = make_prompt_chatbot(query , relevant_text)
    answer = get_openai_resp(system_message ,  chat_history=chat_history , prompt=prompt)
    print(relevant_text)
    print("Answer :- " , answer)
    print('------------------------')
    query2 = "Can you please tell me the design of this ?"
    relevant_text2 = get_relevant_chunk(query2 , vectorsotre)
    prompt2 = make_prompt_chatbot(query2 , relevant_text2)
    print(relevant_text2)
    answer = get_openai_resp(system_message , chat_history , prompt2)
    print("Answer :- " , answer)

In [41]:
main()

('Product name : Superstar Bold 2\n', 'Product brand : Adidas\n', 'Product price : 11999\n', 'Product colour : Black\n', 'Description : Classic basketball shoes with a modern twist and bold design\n')
Answer :-  Assistant : 11999
------------------------
('Product name : Classic Leather\n', 'Product brand : Reebok\n', 'Product price : 4999\n', 'Product colour : Beige\n', 'Description : Retro style everyday wear\n')
Answer :-  Assistant : Reebok


In [44]:
from langgraph.graph import StateGraph,START,END,add_messages
from typing import TypedDict, List , Annotated

In [ ]:
class AgentState(TypedDict):
    messages : Annotated[List , add_messages]
    query : str

In [46]:
from langchain_core.output_parsers import StrOutputParser
llm = ChatOpenAI(api_key=os.getenv("OPEN_AI_API") , model="gpt-4o-mini")

In [47]:
def decision_node(state : AgentState):
    history_chat = state['messages']
    query = state['query']
    context = "\n\n".join(history_chat)
    prompt = f"""I will provide you the context and the question so dont guess or give answer from some
                random your source...if its available in the context then say like its available and give 
                answer of the users question and if not like you think that context is not enough to
                answer then tell go to just pinecone \n\n
                Question : {query}\n\n
                Context : {context}
    """
    chain = prompt | llm | StrOutputParser()
    response = chain.invoke()
    return response

In [49]:
graph = StateGraph(AgentState)

graph.add_node("decision_node",decision_node)
graph.set_entry_point("decision_node")
graph.add_edge("decision_node", END)

In [50]:
app = graph.compile()

In [ ]:
app.invoke({
    
})

## **------------------**

In [56]:
import os 
from typing import TypedDict, List, Annotated
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone 
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

In [57]:
# Env Setup
load_dotenv()
OPENAI_KEY = os.getenv("OPEN_AI_API")
PINECONE_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_KEY)
index_name = 'shop-product-sample'
index = pc.Index(index_name)
embedding_model = OpenAIEmbeddings(
    api_key=OPENAI_KEY,
    model="text-embedding-3-small"
)
vector_store = PineconeVectorStore(
    index = index ,
    embedding= embedding_model,
    text_key="Description"
)

In [60]:
llm = ChatOpenAI(
    api_key=OPENAI_KEY,
    model = "gpt-4o-mini",
    temperature=0
)

In [59]:
# Lang graph state
class ChatState(TypedDict):
    messages : Annotated[List[BaseMessage] , add_messages]
    question : str 
    context : str 
    answer : str 
    need_retrieval : bool 

In [67]:
## Node 1 :- Check History

def history_agent(state : ChatState):
    history = state['messages']
    question = state['question']

    if len(history) == 0:
        return {
            "need_retrieval" : True,
            "answer" : ""
        }
    
    history_text = ""

    for msg in history:
        if isinstance(msg,HumanMessage):
            history_text += (f"User : {msg.content}")
        elif isinstance(msg,AIMessage):
            history_text += (f"Assistant : {msg.content}")

    prompt = f"""
    You are a conversation memory system.
    Previous Conversation:
    {history_text}
    New user Question:
    {question}
    
    INSTRUCTION:
    - If previous conversation contains enough information, answer user directly
    - If previous conversation doesnot contain the answer reply exactly:
        NOT_FOUND

    Answer:
    """

    response = llm.invoke(prompt)
    result = response.content.strip()

    if result == "NOT_FOUND":
        return {
            "need_retrieval" : True,
            "answer" : ""
        }
    
    return {
        "need_retrieval" : False,
        "answer" : result
    }

In [62]:
# Router 
def route_question(state : ChatState):
    if state['need_retrieval']:
        return "pinecone"
    else:
        return "save"

In [63]:
# Node 2 :- Pinecone Search

def pinecone_search(state : ChatState):
    docs = vector_store.similarity_search(state['question'] , k=1)
    context = ""
    for doc in docs:
        context += f"""
    Product :
    {doc.page_content}

    Information:
    {doc.metadata}
    """
        
    return {
        "context" : context
    }

In [64]:
# Node 3 Generate from Pinecone
def generate_from_context(state : ChatState):
    prompt = f""" 
    You are a shopping assistant.
    Use this produt information:
    {state['context']}

    Question:
    {state['question']}

    Answer Clearly
    """ 
    response = llm.invoke(prompt)

    return {
        "answer" : response.content
    }

In [76]:
# Node 4 Save memory context
def save_memory(state : ChatState):
    user_message=f"""

    Question:

    {state["question"]}


    Retrieved Context:

    {state["context"]}

    """

    state['messages'].append(
        HumanMessage(content=user_message)
    )
    state['messages'].append(
        AIMessage(
            content = state['answer']
        )
    )

    return state 

In [70]:
# Graph generation
graph = StateGraph(ChatState)
graph.add_node("history" , history_agent)
graph.add_node("generate" , generate_from_context)
graph.add_node("pinecone" , pinecone_search)
graph.add_node("save" , save_memory)
graph.set_entry_point("history")
graph.add_conditional_edges(
    "history",
    route_question,
    {
        "pinecone" : "pinecone",
        "save" : "save"
    }
)
graph.add_edge("pinecone" , "generate")
graph.add_edge("generate" , "save")
graph.add_edge("save" , END)

In [71]:
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

In [72]:
def chat(user_id , question):
    result = app.invoke({
        "messages" : [],
        "question" : question,
        "context" : "",
        "answer" : "",
        "need_retrieval":False
    } , config={
        "configurable" : {"thread_id" : user_id}
    })
    
    return result['answer']

In [75]:
user_id = "customer_1"

while True:
    query = input("\nUser : ")
    if query == "exit":
        break 
    answer = chat(user_id , query)
    print("\You : " , query)
    print("\nAssistant : " , answer)

<>:8: SyntaxWarning: invalid escape sequence '\Y'
<>:8: SyntaxWarning: invalid escape sequence '\Y'
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_18864\1817840285.py:8: SyntaxWarning: invalid escape sequence '\Y'
  print("\You : " , query)


\You :  What is price of Presto Fly

Assistant :  The price of the Presto Fly running shoes is 8999.
\You :  which brand make this product?

Assistant :  The Presto Fly is made by Nike.
\You :  give me description of that product

Assistant :  The Presto Fly is priced at ₹8,999 and is made by Nike. Unfortunately, I don't have a detailed description of the product.
